# 🔬 Arogya Mantra — Skin Disease Model Training
This notebook trains a MobileNetV2 model on your skin disease dataset from Google Drive and pushes the trained model back to your GitHub repo.

**Steps:**
1. Mount Google Drive
2. Set your GitHub token
3. Run all cells (Runtime → Run All)
4. Done! Model is pushed to GitHub automatically.

## ⚙️ Step 1 — Configuration
Fill in your details below before running.

In [ ]:
# ✏️ FILL THESE IN
GITHUB_TOKEN = "YOUR_GITHUB_TOKEN_HERE"  # Your GitHub PAT
GITHUB_REPO  = "sohamauti9623/arogya-mantra-app"

# Google Drive folder IDs — update these to match YOUR Drive folders
# To get folder ID: open folder in Drive → copy the ID from the URL
# e.g. drive.google.com/drive/folders/THIS_PART_IS_THE_ID
DRIVE_FOLDERS = {
    "herpes_zoster": "1UPpZuqYFFrsdYZ3H_xWXaehTUPMyf9I4",  # Your HZ folder
    # Add other class folders below if you have them:
    # "acne":          "FOLDER_ID_HERE",
    # "eczema":        "FOLDER_ID_HERE",
    # "psoriasis":     "FOLDER_ID_HERE",
    # "chickenpox":    "FOLDER_ID_HERE",
    # "healthy_skin":  "FOLDER_ID_HERE",
}

print('✅ Config set')

## 📁 Step 2 — Mount Google Drive & Clone Repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive mounted')

In [ ]:
import subprocess, os

# Clone the repo
repo_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_REPO}.git"
!git clone {repo_url} /content/repo
os.chdir('/content/repo')
!git config user.email "colab@train.ai"
!git config user.name "Colab Training"
print('✅ Repo cloned')

## 🖼️ Step 3 — Copy Images from Drive into Dataset Folders

In [ ]:
import shutil
from pathlib import Path
import gdown

DATASET_DIR = Path('/content/repo/dataset')
VALID_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

# Download each class folder from Google Drive
for class_name, folder_id in DRIVE_FOLDERS.items():
    dest = DATASET_DIR / class_name
    dest.mkdir(parents=True, exist_ok=True)
    
    print(f'\n📥 Downloading {class_name} from Drive...')
    tmp_dir = f'/content/tmp_{class_name}'
    
    try:
        gdown.download_folder(
            id=folder_id,
            output=tmp_dir,
            quiet=False,
            use_cookies=False
        )
        
        # Copy all valid images into dataset folder
        copied = 0
        for img_path in Path(tmp_dir).rglob('*'):
            if img_path.is_file() and img_path.suffix.lower() in VALID_EXTS:
                shutil.copy2(img_path, dest / img_path.name)
                copied += 1
        
        print(f'  ✅ {copied} images copied to dataset/{class_name}/')
        shutil.rmtree(tmp_dir, ignore_errors=True)
        
    except Exception as e:
        print(f'  ❌ Failed: {e}')

# Show final counts per class
print('\n📊 Dataset summary:')
for cls_dir in sorted(DATASET_DIR.iterdir()):
    if cls_dir.is_dir():
        imgs = list(cls_dir.rglob('*.jpg')) + list(cls_dir.rglob('*.jpeg')) + list(cls_dir.rglob('*.png'))
        print(f'  {cls_dir.name}: {len(imgs)} images')

## 🤖 Step 4 — Install Dependencies & Train Model

In [ ]:
!pip install -q scikit-learn pillow tensorflow

In [ ]:
"""Training pipeline — adapted from backend/ml/train_model.py"""
import json, pickle
import numpy as np
import tensorflow as tf
from PIL import Image, UnidentifiedImageError
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from pathlib import Path

DATASET_DIR   = Path('/content/repo/dataset')
ARTIFACTS_DIR = Path('/content/repo/backend/ml/artifacts')
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE       = (224, 224)
BATCH_SIZE       = 32
EPOCHS           = 15
VALIDATION_SPLIT = 0.2
SEED             = 42

# Only include classes that actually have images
ALL_CLASSES = ['acne', 'eczema', 'psoriasis', 'chickenpox', 'herpes_zoster', 'healthy_skin']
DISPLAY_NAMES = {
    'acne': 'Acne', 'eczema': 'Eczema', 'psoriasis': 'Psoriasis',
    'chickenpox': 'Chickenpox', 'herpes_zoster': 'Herpes Zoster', 'healthy_skin': 'Healthy Skin'
}

CLASS_NAMES = [
    c for c in ALL_CLASSES
    if (DATASET_DIR / c).exists() and
       len(list((DATASET_DIR / c).rglob('*.jpg')) +
           list((DATASET_DIR / c).rglob('*.jpeg')) +
           list((DATASET_DIR / c).rglob('*.png'))) >= 10
]
print(f'Training on classes: {CLASS_NAMES}')

# Remove corrupted images
removed = 0
for img_path in DATASET_DIR.rglob('*'):
    if not img_path.is_file() or img_path.suffix.lower() not in {'.jpg','.jpeg','.png','.bmp','.webp'}:
        continue
    try:
        with Image.open(img_path) as img:
            img.verify()
    except Exception:
        img_path.unlink(missing_ok=True)
        removed += 1
print(f'Removed {removed} corrupted images')

# Build datasets
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR, labels='inferred', label_mode='categorical',
    class_names=CLASS_NAMES, image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE, shuffle=True,
    validation_split=VALIDATION_SPLIT, subset='training', seed=SEED
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR, labels='inferred', label_mode='categorical',
    class_names=CLASS_NAMES, image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE, shuffle=False,
    validation_split=VALIDATION_SPLIT, subset='validation', seed=SEED
)

# Class weights for imbalanced data
y_indices = np.concatenate([np.argmax(y.numpy(), axis=1) for _, y in train_ds])
classes   = np.unique(y_indices)
weights   = compute_class_weight('balanced', classes=classes, y=y_indices)
class_weights = {int(k): float(v) for k, v in zip(classes, weights)}
print('Class weights:', class_weights)

# Build model
data_aug = tf.keras.Sequential([
    layers.RandomRotation(0.11),
    layers.RandomFlip('horizontal'),
    layers.RandomZoom(height_factor=(-0.2, 0.2), width_factor=(-0.2, 0.2)),
    layers.RandomBrightness(0.2),
], name='data_augmentation')

inputs     = layers.Input(shape=(224, 224, 3))
x          = data_aug(inputs)
x          = layers.Rescaling(1.0 / 255)(x)
base_model = MobileNetV2(include_top=False, weights='imagenet', input_shape=(224, 224, 3))
base_model.trainable = False
x          = base_model(x, training=False)
x          = layers.GlobalAveragePooling2D()(x)
x          = layers.Dropout(0.25)(x)
outputs    = layers.Dense(len(CLASS_NAMES), activation='softmax')(x)
model      = models.Model(inputs=inputs, outputs=outputs)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

print(f'\n🚀 Starting Phase 1 training ({len(CLASS_NAMES)} classes)...')
ckpt_path = str(ARTIFACTS_DIR / 'best_checkpoint.keras')
callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=4, restore_best_weights=True),
    ModelCheckpoint(ckpt_path, monitor='val_accuracy', save_best_only=True)
]

train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
val_ds   = val_ds.prefetch(tf.data.AUTOTUNE)

model.fit(train_ds, validation_data=val_ds, epochs=10,
          class_weight=class_weights, callbacks=callbacks)

# Fine-tuning phase
print('\n🔧 Phase 2: Fine-tuning...')
base_model.trainable = True
for layer in base_model.layers[:-20]:
    layer.trainable = False
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS,
          class_weight=class_weights, callbacks=callbacks)

# Save artifacts
model.save(ARTIFACTS_DIR / 'skin_disease_model.h5')
class_mapping = {idx: DISPLAY_NAMES.get(name, name) for idx, name in enumerate(CLASS_NAMES)}
(ARTIFACTS_DIR / 'class_mapping.json').write_text(json.dumps(class_mapping, indent=2))
(ARTIFACTS_DIR / 'label_encoder.pkl').write_bytes(pickle.dumps(CLASS_NAMES))
print('\n✅ Model saved to backend/ml/artifacts/')
print('Class mapping:', class_mapping)

## 🚀 Step 5 — Push Trained Model to GitHub

In [ ]:
import subprocess
os.chdir('/content/repo')

# Configure git LFS for large model file
!git lfs install
!git lfs track "*.h5"
!git lfs track "*.keras"

# Stage and commit
!git add backend/ml/artifacts/
!git add .gitattributes
!git status

result = subprocess.run(
    ['git', 'commit', '-m',
     'feat: add trained MobileNetV2 model with herpes zoster dataset\n\n'
     '- Trained on dermnet dataset with herpes_zoster class\n'
     '- MobileNetV2 backbone with fine-tuning\n'
     '- Class weights balanced for imbalanced dataset'],
    capture_output=True, text=True
)
print(result.stdout)
print(result.stderr)

push = subprocess.run(['git', 'push', 'origin', 'main'], capture_output=True, text=True)
print(push.stdout)
print(push.stderr)

if push.returncode == 0:
    print('\n✅ SUCCESS! Model pushed to GitHub.')
    print(f'View at: https://github.com/{GITHUB_REPO}/tree/main/backend/ml/artifacts')
else:
    print('\n⚠️ Push failed. Try running: git push origin main')

## ✅ Done!
Your trained model is now on GitHub. The backend will automatically use it on the next deploy.

If you want to retrain later with more data, just add images to the Drive folders and run this notebook again.